## __1. Multiclass Classification__

Multiclass classification is a type of classification task that involves classifying instances into one of three or more classes. Unlike binary classification, which deals with two classes, multiclass classification can handle problems where an instance can belong to multiple categories.

**Example:** A tree can be classified as a banyan tree, a palm tree, a pine tree, or an oak tree.

Some of the popular algorithms used for multi-class classification are:

- Naive Bayes
- K-Nearest Neighbors
- Decision Trees


Lets take a multiclass problem where we can address all the Multiclass algorithms and will see the clear cut difference in the model performance:

## __1.1 **Example** with Online Gaming Behavior Dataset:__

The **Online Gaming Behavior** dataset on Kaggle is designed to analyze and predict players' behaviors in online gaming. It includes data on various aspects such as player identification, session duration, in-game actions, and purchases. The dataset comprises several features that capture player activities and interactions within the game environment, providing a rich source of information for developing predictive models. This data can be leveraged to understand player engagement, forecast future behavior, and tailor strategies to enhance user experience and retention.

**Number of Instances:** 40034

**Number of Attributes:** 13

**Attribute Information:**

- **PlayerID:** Unique identifier for each player.
- **Age:** Age of the player.
- **Gender:** Gender of the player.
- **Location:** Geographic location of the player.
- **GameGenre:** Genre of the game the player is engaged in.
- **PlayTimeHours:** Average hours spent playing per session.
- **InGamePurchases:** Indicates whether the player makes in-game purchases (0 = No, 1 = Yes).
- **GameDifficulty:** Difficulty level of the game.
- **SessionsPerWeek:** Number of gaming sessions per week.
- **AvgSessionDurationMinutes:** Average duration of each gaming session in minutes.
- **PlayerLevel:** Current level of the player in the game.
- **AchievementsUnlocked:** Number of achievements unlocked by the player.

**Target Variable**
- **EngagementLevel:** Categorized engagement level reflecting player retention (Medium, High, and Low).



**Note:** We will use this dataset to explore and compare various multiclass algorithms, examining how their performance varies depending on the type and implementation of each algorithm.

Let's first start with the implementation of **Naive Bayes**, but make sure to conduct proper data preprocessing to make it suitable for modeling.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import label_binarize
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.metrics import classification_report, confusion_matrix
from itertools import cycle


In [ ]:
# Load the dataset
data_df = pd.read_csv('online_gaming_behavior_dataset.csv')
data_df.head()

In [ ]:
data_df.shape

In [ ]:
# Checking for missing values
data_df.isnull().sum()

In [ ]:
# Statistical summary of the dataset
data_df.describe()


In [ ]:
# Distribution of the target variable
data_df['EngagementLevel'].value_counts()

In [ ]:
target_column = 'EngagementLevel'
X = data_df.drop(columns=[target_column])
y = data_df[target_column]

In [ ]:
# Convert categorical string variables to numerical values
categorical_columns = ['Gender', 'Location', 'GameGenre', 'GameDifficulty']
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

# Create dummies for other categorical variables if any
#X = pd.get_dummies(X, drop_first=True)

In [ ]:
# Heatmap of the correlation matrix using seaborn library
plt.figure(figsize=(12, 8))
correlation_matrix = X.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

In [ ]:
X_train

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

## __1.2 Applying Naive Bayes Algorithm on Online Gaming Behavior dataset__

In [ ]:
# Import Required Libraries and apply Naive Bayes algorithm
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train_sc, y_train)

# Predict on testing set
y_pred_nb = nb_model.predict(X_test_sc)
y_pred_prob_nb = nb_model.predict_proba(X_test_sc)

In [ ]:
# Determine the class order
class_order = nb_model.classes_
print("Class order:", class_order)

# Create a DataFrame to display actual and predicted labels with probabilities
results_df = pd.DataFrame({
    'Actual Label': y_test,
    'Predicted Label': y_pred_nb,
})

# Add predicted probabilities with correct class names
for i, class_name in enumerate(class_order):
    results_df[f'Predicted Probability {class_name}'] = np.round(y_pred_prob_nb[:, i], 2)

# Display the first few rows of the results DataFrame
results_df.head()


In [ ]:
print("\nNaive Bayes Classifier:")
training_accuracy =  accuracy_score(y_train, nb_model.predict(X_train_sc))
testing_accuracy = accuracy_score(y_test, y_pred_nb)

print(f"Training Accuracy: {training_accuracy}")
print(f"Testing Accuracy: {testing_accuracy}")

In [ ]:
# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_nb, labels=class_order)
print("Confusion Matrix:")
print(conf_matrix)

# Display the confusion matrix with labels
conf_matrix_df = pd.DataFrame(conf_matrix, index=class_order, columns=class_order)
print("Confusion Matrix with Class Labels:")
print(conf_matrix_df)


#### __Observation__

The confusion matrix is a 3x3 matrix because it is a multiclass classification problem with three classes (Medium, High, Low).

**High:**

This class has a relatively high number of true positives (3234), but also a considerable number of false negatives, indicating that some instances misclassified as Low and medium

**Low:**

This class has a moderate number of true positives (2854), but a higher number of false negatives, indicating frequent misclassification of `low` instances as `high` or `medium`.

**Medium:**

This class has a very high number of true positives (7419), indicating the model is very good at correctly identifying `medium` instances.

In [ ]:
# Print classification report
Class_report = classification_report(y_test, y_pred_nb)
print("Naïve Bayes Classification Report:")
print(Class_report)

- `label_binarize` is a function from Scikit-Learn that is used to convert class labels into a binary format suitable for certain types of classifiers and evaluation metrics.
- This is particularly useful when you need to handle multiclass problems with methods that are inherently designed for binary classification, such as calculating ROC curves and AUC scores.

Plot the ROC curve for each class separately in a multiclass setting.

In [ ]:
# Check if more than one class is present in y_test
if len(set(y_test)) > 1:
    # Plot ROC curve
    y_test_binarized = label_binarize(y_test, classes=nb_model.classes_)
    n_classes = y_test_binarized.shape[1]

    y_score = nb_model.predict_proba(X_test_sc)

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
        roc_auc[i] = roc_auc_score(y_test_binarized[:, i], y_score[:, i])

    # Plotting the ROC curves
    plt.figure()
    colors = cycle(['aqua', 'darkorange', 'cornflowerblue'])
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2, label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.show()
else:
    print("ROC AUC score is not defined as only one class is present in y_true.")

#### __Observation__

1. Class 0/High: The model performs very well in classifying instances of Class 0. An AUC of 0.94 indicates that there is a 94% chance that the model will correctly distinguish between a randomly chosen positive instance and a randomly chosen negative instance for Class 0.

2. Class 1/Low: The model has excellent performance for Class 1, with an AUC of 0.91. This means the model has a 91% chance of correctly distinguishing between a positive instance and a negative instance for Class 1.

3. Class 2/Medium: The model performs very well for Class 2, with an AUC of 0.90. This indicates a 90% chance of the model correctly distinguishing between a positive instance and a negative instance for Class 2.

Diagonal Line (Black dashed line):

The diagonal line represents random guessing, with an AUC of 0.5. Any ROC curve below this line indicates worse than random performance, while curves above this line indicate better than random performance.

**Key Observations:**

The Naive Bayes performs exceptionally well for all classes, with AUC values ranging from 0.90 to 0.94. The model is best at classifying instances of Class 0, followed closely by Class 1, and then Class 2.



## __1.3 Applying K-Nearest Neighbors on Online Gaming Behavior Dataset__

In [ ]:
#Import Required Libraries and apply K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_sc, y_train)

# Predict on testing set
y_pred_knn = knn_model.predict(X_test_sc)
y_pred_prob_knn = knn_model.predict_proba(X_test_sc)

In [ ]:
# Determine the class order
class_order = knn_model.classes_
print("Class order:", class_order)

# Create a DataFrame to display actual and predicted labels with probabilities
results_df = pd.DataFrame({
    'Actual Label': y_test,
    'Predicted Label': y_pred_knn,
})

# Add predicted probabilities with correct class names
for i, class_name in enumerate(class_order):
    results_df[f'Predicted Probability {class_name}'] = np.round(y_pred_prob_knn[:, i], 2)

# Display the first few rows of the results DataFrame
results_df.head()

In [ ]:
# Calculate training and testing accuracy
print("\nK-Nearest Neighbors Classification")
training_accuracy =  accuracy_score(y_train, knn_model.predict(X_train_sc))
testing_accuracy = accuracy_score(y_test, y_pred_knn)

print(f"Training Accuracy: {training_accuracy}")
print(f"Testing Accuracy: {testing_accuracy}")

In [ ]:
# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_knn, labels=class_order)
print("Confusion Matrix:")
print(conf_matrix)

# Display the confusion matrix with labels
conf_matrix_df = pd.DataFrame(conf_matrix, index=class_order, columns=class_order)
print("Confusion Matrix with Class Labels:")
print(conf_matrix_df)


#### __Observation__

**High:**

This class has a moderate number of true positives (3049), but also a significant number of false negatives, indicating that many `high` instances are misclassified as `medium` or `low`.

**Low:**

This class has a lower number of true positives (2360) and a high number of false negatives, indicating frequent misclassification of `Low` instances as `medium` or `high`.

**Medium:**

This class has a high number of true positives (6028), indicating the model is relatively good at identifying `Medium` instances. However, there is still a considerable number of false negatives.

In [ ]:
# Print classification report
Class_report = classification_report(y_test, y_pred_knn)
print("K-Nearest Neighbors Classification Report:")
print(Class_report)

Plot the ROC curve for each class separately in a multiclass setting.

In [ ]:
# Check if more than one class is present in y_test
if len(set(y_test)) > 1:
    # Plot ROC curve
    y_test_binarized = label_binarize(y_test, classes=knn_model.classes_)
    n_classes = y_test_binarized.shape[1]

    y_score = knn_model.predict_proba(X_test_sc)

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
        roc_auc[i] = roc_auc_score(y_test_binarized[:, i], y_score[:, i])

    # Plotting the ROC curves
    plt.figure()
    colors = cycle(['aqua', 'darkorange', 'cornflowerblue'])
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2, label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.show()
else:
    print("ROC AUC score is not defined as only one class is present in y_true.")

#### __Observation__

**Class 0 (Cyan Line)**

Area Under the Curve (AUC): 0.89

The model performs well in classifying instances of Class 0/Medium. An AUC of 0.89 indicates that there is an 89% chance that the model will correctly distinguish between a randomly chosen positive instance and a randomly chosen negative instance for Class 0.

**Class 1 (Orange Line)**

Area Under the Curve (AUC): 0.83

The model has good performance for Class 1, with an AUC of 0.83. This means the model has an 83% chance of correctly distinguishing between a positive instance and a negative instance for Class 1.

**Class 2 (Blue Line)**

Area Under the Curve (AUC): 0.79

The model performs moderately well for Class 2, with an AUC of 0.79. This indicates a 79% chance of the model correctly distinguishing between a positive instance and a negative instance for Class 2.

**Key Observation:**

- The k-NN classifier performs reasonably well for all classes, with AUC values ranging from 0.79 to 0.89.
- If the dataset has imbalanced classes (i.e., some classes have significantly more instances than others), this could affect the performance of the model and the ROC curves. It's essential to consider class distribution when interpreting these results.

## __1.4 Applying Decision Tree on Online Gaming Behavior Dataset__


In [ ]:
#Import Required Libraries and apply decision tree
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_sc, y_train)

# Predict on testing set
y_pred_dt = dt_model.predict(X_test_sc)
y_pred_prob_dt = dt_model.predict_proba(X_test_sc)

In [ ]:
# Determine the class order
class_order = dt_model.classes_
print("Class order:", class_order)

# Create a DataFrame to display actual and predicted labels with probabilities
results_df = pd.DataFrame({
    'Actual Label': y_test,
    'Predicted Label': y_pred_dt,
})

# Add predicted probabilities with correct class names
for i, class_name in enumerate(class_order):
    results_df[f'Predicted Probability {class_name}'] = np.round(y_pred_prob_dt[:, i], 2)

# Display the first few rows of the results DataFrame
results_df.head()

In [ ]:
## Calculate training and testing accuracy
print("\nDecision Tree Classifier:")
training_accuracy =  accuracy_score(y_train, dt_model.predict(X_train_sc))
testing_accuracy = accuracy_score(y_test, y_pred_dt)

print(f"Training Accuracy: {training_accuracy}")
print(f"Testing Accuracy: {testing_accuracy}")

#### __Observation__

- The high training accuracy combined with the slightly lower testing accuracy typically points towards an overfitting scenario.

- In practice, this might mean that the decision tree has complex branches that perfectly classify the training data but slightly miss when predicting new, unseen data.

- Adjustments such as pruning the tree, setting a maximum depth, or increasing the minimum samples required for a node split might help in reducing overfitting and improving the model's generalization.

- Decision trees are prone to overfitting, especially with complex datasets having many features and deep trees. -

- Cross-validation helps in checking whether the model just memorizes the training data or if it generalizes well over unseen data.

In [ ]:
# Cross-validation
from sklearn.model_selection import cross_val_score
scores = cross_val_score(dt_model, X_train_sc, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Mean cross-validation score:", scores.mean())


**Cross-Validation Scores:**

 Cross-validation scores vary slightly but generally remain high, with a mean cross-validation score of 0.82. This indicates good model performance and stability.

In [ ]:
# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_dt, labels=class_order)
print("Confusion Matrix:")
print(conf_matrix)

# Display the confusion matrix with labels
conf_matrix_df = pd.DataFrame(conf_matrix, index=class_order, columns=class_order)
print("Confusion Matrix with Class Labels:")
print(conf_matrix_df)

In [ ]:
# Print classification report
Class_report = classification_report(y_test, y_pred_dt)
print("Decision Trees Classification Report:")
print(Class_report)

The confusion matrix and classification report show that while the model performs well overall, there are some misclassifications